# 分子動力学計算（解析）

In [ ]:
from ase.io import read, write

from ase import units
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary
from ase.md.verlet import VelocityVerlet
from ase.md.npt import NPT
from ase.md import MDLogger

from ase.calculators.lj import LennardJones
from ase.visualize import view
from ase.optimize import BFGS
from ase.phonons import Phonons

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import math

<div class="alert-info">

**Goal of this tutorial**  
The goal of this tutoal is to understand the overview of Molecular Dynamics (MD).  
  
The computational target is very simple system, liquid Argon, but the methodology of the time evolution, microscopic analysis, statistical mechanics-based analysis, and computation of transport phoenomena is widely used in cutting-edge MD research.  
  
This Jupyter notebook provides the MD simulation and the post analysis code based on ASE (Atomic Simulation Environment).  
ASE is one of the most famaous and useful library of Molecular Simulation. ASE supports a wide range of MD software (Classical MD: GROMACS, LAMMPS, Amber, ..., DFTMD: VASP, CP2K, Quantum Espresso, ..., and other post analysis code: Pymatgen). Furthermore, cutting-edge computational method like NeuralNetwork Potential has been developed on ASE. Therefore, the analysis code and the know-how of ASE can be used in any molecular simulations.
</div>

## 1. Equation of motion
The most important equation of MD is Equation of motion (Newton's 2nd law) because it connects the movements of a particle with the mechanics: 

$$m\frac{d^2r(t)}{dt^2} = F = - \frac{\partial U(r)}{\partial r},$$

where $m$ is the mass, $r$ is the position, $F$ is the force and $U(r)$ is the potential energy, which depends only on the position of the particle. This equation determine the dynamic evalution of the position if one know the force acting on the particle. However, most cases cannot be solved analytically, so it should be solved numerically. In addition, the accuracy of the potential energy $U(r)$ is also concern. In summary, first of all,  researcher have to choose the suitable methods for the target. 
- Integrator
    - velocity-Verlet
    - Leap-frog
    - (Predictor-corrector method) not major now
- Potential
    - Classical Force field (FF)    / Length: –10 $^3$ nm, Time: –10 $^1$ μs, approximately 100 ns/day
    - Density Functional Theory (DFT) / Length: –10 $^1$ nm
    - Neural Network Potential (NNP) / Length: –10 $^2$ nm

Especially, the choise of potential (energy calculation) is important. Classical force field is fast but low accurate bacause it relys on empirical parameter and function type. On the other hand, DFT has good accurary but the cost is so expensive. To overcome these limitation, many methodology development and application of Neural Network Potential has become active recently. Neural Network Potential (NNP) learn the DFT calculation, allowing it to perform DFT accuracy calculations much faster.


Of course, this ordinary differential equations is second order one, so the following two initial condition is required for the simulation. 
- Initial Condition
    - Position (Configuration)
    - Velocity

The sequence of MD calculations is as follows.  
  
<img src="./images/Procedure.png" width="30%">


## 2. Liquid Argon

In this section, we get the initial structure of molecular dynamics by following procedures:
1. Read the .xyz file as `ase.Atoms` object
   - Keyword: File format, Periodic boundary condition (PBC)
2. Geometry optimization to get stable structure as initial structure of MD

In [ ]:
# System Setup
argon_atoms = read("argon.xyz")   # read the initial configuration from xyz file
# view(argon_atoms)   # visualization
sigma = 3.4 # Angstrom
epsilon = 0.01034 # eV
argon_lj = LennardJones(epsilon=epsilon, sigma=sigma, rc=7.5)
argon_atoms.calc = argon_lj

MD simulations normally use "Periodic Boundary Condition (PBC)". 
- Remove surface effects and treat system as bulk
- Cutoff of Lennard–Jones potential should be less than half length of the lattice length to avoid a particle interacting with its own image

In [ ]:
argon_atoms.pbc = True  # set periodic boundary conditions

The geometry optimization is performed. An energy minimization procedure consists of adjusting the coordinates of the atoms that are too close to each other until one of the stopping criteria is reached. ASE implements some optimization algorithm, e.g., BFGS, LBFGS, and FIRE. Here, BFGS is used. 

In [ ]:
opt = BFGS(argon_atoms, trajectory="argon_opt.traj")
opt.run(fmax=0.03)  # Criteria is "fmax < 0.03 eV/Angstrom"

## 3. _NVT_ integrator (velocity-Verlet + Nose-Hoover thermostat)

There are many ways to control the system temperature.  
- Berendsen thermostat
- Nose-Hoover thermostat
- Langevin thermostat
- ....  

One of the most commonly used algorithm is **Nose-Hoover thermostat**. Advantage of Nose-Hoover thermostat is that system follows canonical ensemble (*NVT* ensemble). ***NVT*** means constant  <u>***N***</u>umber of particles, <u>***V***</u>olume, and <u>***T***</u>empearture.  
  
Under the canonical (*NVT*) ensemble, the distribution function $f(\boldsymbol{q}, \boldsymbol{p})$ must follow as below relationship:  

$$
f(\boldsymbol{q}, \boldsymbol{p}) = \frac{1}{N!h^{3N}} \frac{\exp(-\beta \mathcal{H}(\boldsymbol{q}, \boldsymbol{p}))}{Z},  \tag{5.1}  
$$


$$
\mathrm{s.t.}\;  Z = \frac{1}{N!h^{3N}}\int \rm{d}{\boldsymbol{q}}\int \rm{d}{\boldsymbol{p}} \exp \left[ -\beta \mathcal{H} \right], 
$$

$$
\beta = \frac{1}{k_{\rm{B}}T},
$$

where $\boldsymbol{q}$, $\boldsymbol{p}$, $N$, $h$, $k_{\rm{B}}$, and $T$ are position, momentum, number of particles, Planck constant, Boltzmann constant, and temperature, respectively. $Z$ is partition function and $\beta$ is inverse temperature. Under the canonical ensemble, one physical property $A(\boldsymbol{q}, \boldsymbol{p})$ is obtained as follows:  

$$
\langle A \rangle _{NVT} = \int \rm{d}{\boldsymbol{q}}\int \rm{d}{\boldsymbol{p}} \; \it{A}(\boldsymbol{q}, \boldsymbol{p})f(\boldsymbol{q}, \boldsymbol{p}).   \tag{5.2}
$$

By using Nose-Hoover thermostat, system follows the Eq.(5.1). Thus, if the simulation time is enough long, the time averaged $ A(\boldsymbol{q}, \boldsymbol{p}) $ is equal to Eq.(5.2):  

$$
\langle A \rangle _{NVT} = \langle A \rangle _{\rm{time, Nose-Hoorver}} = \frac{\sum_{k}^M A(t_{k})}{M}
$$ 

The equation of motion with Nose-Hoover thermostat is given by

$$\dot{q_i} = \frac{p_i(t)}{m_i} , $$

$$\dot{p_i} = F_i(t) - \zeta p_i   ,$$

$$\dot{\zeta} = \frac{1}{\tau_{T}^2} \left[ \frac{\mathcal{T}(t)}{T_0s} - 1 \right] , $$

$$\mathrm{s.t.} \; \mathcal{T}(t) = \frac{1}{3 Nk_{\rm{B}}} \sum_{i}^N m_iv_{i}^2  ,$$

where $\mathcal{T}(t)$ is called instantaneous temperature and $\tau_{T}$ is the time constant of Nose-Hoover thermostat.  
`Notice`: Instantaneous temperature has no physical meaning. Only the ensemble averaged of $\mathcal{T}(t)$, i.e., $\langle \mathcal{T} \rangle = T $ (Temperature) has physical meaning.  
Users should decide proper $\tau_{T}$. According to the [ASE documents](https://wiki.fysik.dtu.dk/ase/ase/md.html), the good choice of $\tau_{T}$ (parameter name: ttime) is 25 fs. 

In [ ]:
# argon_atoms = mdtraj[-1].copy()
# argon_atoms.calc = argon_lj

integrator = NPT(argon_atoms, 5 * units.fs, temperature_K = 94.4, ttime = 25 * units.fs, pfactor = None, trajectory="md_nvt.traj", loginterval=10) # NH
integrator.attach(MDLogger(integrator, argon_atoms, logfile="md_nvt.log", header=True, stress=False,
                mode="w"), interval=10)
nsteps = 20000
def progress():
    import datetime
    global initial_timestamp
    global initial_step
    try: 
        time_consumed = datetime.datetime.now() - initial_timestamp
        # 1stepあたりにかかった実時間 realtime_per_step
        realtime_per_step = time_consumed.total_seconds() / (integrator.get_number_of_steps() - initial_step)
        # 残り時間
        remaining_time = (nsteps - integrator.get_number_of_steps()) * realtime_per_step
        print(f"Step: {integrator.get_number_of_steps()} / {nsteps}, T: {argon_atoms.get_temperature():.2f} K, Remaining time: {remaining_time:.2f} s")
    except:
        initial_timestamp = datetime.datetime.now()
        initial_step = integrator.get_number_of_steps()
        print(f"Step: {integrator.get_number_of_steps()} / {nsteps}, T: {argon_atoms.get_temperature():.2f} K")
integrator.attach(progress, interval=100)

# Execute MD
try:
    del globals()["initial_timestamp"]
    del globals()["initial_step"]
except:
    pass
integrator.run(nsteps)

In [ ]:
df = pd.read_csv("md_nvt.log", sep=r'\s+')
df

In [ ]:
df.describe()

In [ ]:
df.plot(subplots=True, layout=(3,2), x="Time[ps]",figsize=(10,6))

## 4. Post analysis

The following physical properties are calculated.  
For all analyses, the initial 5000 fs = 5 ps is not used in the analysis due to the equilibration of the system. 5000 fs and beyond are used in the analysis as the results of the production run.

**Table of contents**
- [Radial distribution function](#rdf) 
- [Mean-squared displacement & Diffusion coefficient](#msd)
- [Velocity autocorrelation function & Vibrational density of states](#vacf)

### 4.1. Radial distribution function (RDF, g of r, g(r)) <a id='rdf'></a>

In [ ]:
! conda install -c conda-forge MDAnalysis -y  # install MDAnalysis for RDF calculation

In [ ]:
from ase.io import write

mdtraj = read("md_nvt.traj", index=":")
mdtraj = mdtraj[int(len(mdtraj)*0.2):]  # use the last 80% of the trajectory for RDF calculation  
write("mdtraj.pdb",mdtraj)

In [ ]:
import MDAnalysis
import MDAnalysis.analysis.rdf as mda

In [ ]:
u = MDAnalysis.Universe("mdtraj.pdb", "mdtraj.pdb")

In [ ]:
u_select1 = u.select_atoms(f"element Ar")
u_select2 = u.select_atoms(f"element Ar")

rmax = 10
dr = 0.01
rdf = mda.InterRDF(u_select1, u_select2, range=(0,rmax), nbins=int(rmax/dr))
rdf.run()
# 0/0 devision error occurs at r=0, so we ignore the first bin
r = rdf.results.bins[1:]
g = rdf.results.rdf[1:]

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

params = {
    "font.size": 11.5,
    "font.family": "Arial",
    "axes.labelsize": "large",
    "axes.linewidth": 1.2,
    "xtick.major.size": 7,    # major tick size in points
    "xtick.minor.size": 4,     # minor tick size in points
    "xtick.major.width": 1.0,     # major tick width in points
    "xtick.minor.width": 1.0,     # minor tick width in points
    "xtick.major.pad": 5.0,
    "xtick.minor.pad": 5.0,     # distance to the minor tick label in points
    "xtick.direction" : "in",
    "ytick.right" : True,
    "ytick.major.size": 7,     # major tick size in points
    "ytick.minor.size": 4,       # minor tick size in points
    "ytick.major.width": 1,     # major tick width in points
    "ytick.minor.width": 1.0,    # minor tick width in points
    "ytick.major.pad": 5,     # distance to major tick label in points
    "ytick.minor.pad": 5,    # distance to the minor tick label in points
    "ytick.direction": "in",
    "legend.fontsize": 10,
    "figure.figsize": (3.0, 2.25),
    "axes.prop_cycle": plt.cycler(color=['#e41a1c', '#377eb8', '#000000', '#984ea3',
                                         '#ff7f00', '#4daf4a', '#F6BE00', '#999999',
                                         '#ffff33', '#a65628', '#8B8000','#f781bf'])
}
plt.rcParams.update(params)

fig, ax = plt.subplots()
ax.xaxis.set_minor_locator(ticker.AutoMinorLocator(2))
ax.yaxis.set_minor_locator(ticker.AutoMinorLocator(2))

ax.plot(r, g, label="Ar-Ar")
ax.set_xlabel("$r$ (${\mathrm {\AA}}$)")
ax.set_ylabel("$g$($r$)")
ax.legend()

### 5.2. Mean-squared displacement (MSD) + Self-diffusion Coefficient <a id="msd"></a>

Mean-squared displacement (MSD) is often calculated to obtain self-diffusion coefficient. The higher self-diffusion coefficient is, the more the particle moves easily.  

In the field of electrochemistry or battery, self-diffusion coefficient plays an important role. For example, in a Li-ion battery, it is desirable for lithium to move at high speed inside. To quantify the high speedness, self-diffusion coefficient is often used as the indicator. Experimentally, self-diffusion coefficient is measured by PFG-NMR. 
  
<u> **Note** </u>  
Sometimes, Li-ion self-diffusion coefficient $D_{\rm{self}}^{\rm{Li}}$ is treated as Li-ion conductivity $\sigma _{\rm{Li}}$ by Nernst-Einstein approximation as follows: 
$$
\sigma_{\rm{Li}} \simeq \frac{e^2}{Vk_{\rm{B}}T}{N_{\rm{Li}}}z_{\rm{Li}}^2D_{\rm{self}}^{\rm{Li}}. 
$$
However, this approximation is valild only in dilute electrolyte. Therefore, it should be noted that this approximation breaks down for high concentration electrolytes and solid electrolytes.

----
Self-diffusion coefficient, $D_{\rm{self}}$ is obtained by the following equation:

$$
D_{\rm{self}} = \frac{1}{2dt} \lim_{t \to \infty} \langle |\boldsymbol{r}(t) - \boldsymbol{r}(0)|^2 \rangle  ,
$$

where *d* is the dimension of $\boldsymbol{r}(t)$. The right-hand side $ \langle |\boldsymbol{r}(t) - \boldsymbol{r}(0)|^2 \rangle $ is called mean-squared displacement (MSD).   
The ensemble average of MSD is a little tricky. 

$$
\begin{align}
\mathrm{MSD}(t) & = \langle |\boldsymbol{r}(t) - \boldsymbol{r}(0)|^2 \rangle \nonumber \\
                & = \left\langle \left[ \sum_i^N (r_i(t)-r_i(0))^2 \right]/N \right\rangle  \nonumber
\end{align}
$$

The important point is the angled brackets represent the average for different initial times or independent runs. To improve the statistical accuracy, one can prepare *n* arrays of data sequence with slightly shifted initial time of $\tau$ values and the data for N arrays were averaged like this 

$$
% \langle |r_{i}(t) - r_i(0)|^2 \rangle  = \sum_{t=0}^{n-1}\frac{1}{n-t}\sum_{\tau=1}^{n-t}(\boldsymbol{r}_i(t+\tau)-\boldsymbol{r}_i(\tau))^2  .
\langle |r_{i}(t) - r_i(0)|^2 \rangle  = \frac{1}{n-t}\sum_{\tau=0}^{n-t-1}(\boldsymbol{r}_i(t+\tau)-\boldsymbol{r}_i(\tau))^2  .
$$

The code immediately below is a direct implementation of the MSD calculation.  
However, this code is very slow due to $\mathcal{O}(N^2)$......

> <u>**Note**</u>  
> When calculating MSD (or similar displacements), the **unwrapped** coordinate is needed.  
> Under the periodic boundary condition, some MD softwares give the wrapped coordinate, where the all coordinates is put inside the simulation cell.    
> In this case, converting the wrapped coordinate to unwrapped one is necessary for MSD.  
> The following code simply quantify the displacement between the two consecutive snapshot by minimum image convention.  
> If the displacement between two consecutive steps is greater than half the cell length of the simulation, the displacement obtained by this code will be incorrect. No particle must move more than cell length between consecutive steps.  
> Trajectory generated by ASE is unwrapped one.  
>   
> <img src="./images/wrap_unwrap.png" width="40%">  
>
>  
> <u> Additional comment </u>  
> Please be careful of trajectory obtained by *NpT* simulation. 
> My following code will give incorrect MSD in the case of *NpT*. 
> Please read https://pubs.acs.org/doi/10.1021/acs.jctc.3c00308

In [ ]:
mdtraj = read("md_nvt.traj", index=":")
mdtraj = mdtraj[int(len(mdtraj)*0.2):]  # use the last 80% of the trajectory for RDF calculation  

In [ ]:
from ase.geometry import find_mic

# Create wrapped trajectory
mdtraj_wrap = mdtraj.copy()
for traj in mdtraj:
    traj.wrap()
view(mdtraj_wrap)

In [ ]:
# wrapped trajectory: mdtraj_wrap -> unwrapped trajectory
mdtraj_unwrap = mdtraj_wrap.copy()
diff_ary = np.zeros((len(mdtraj_wrap), len(mdtraj_wrap[0]), 3))
for i_trj in range(1, len(mdtraj_wrap)):
    diffs = mdtraj_unwrap[i_trj].positions - mdtraj_unwrap[i_trj-1].get_positions()
    for i, d in enumerate(diffs):
        diffs[i],_ = find_mic(d, mdtraj_unwrap[i_trj].cell, True)
    diff_ary[i_trj] = diffs

for i_trj in range(1, len(mdtraj_wrap)):
    mdtraj_unwrap[i_trj].positions = mdtraj_unwrap[i_trj-1].positions + diff_ary[i_trj]


In [ ]:
view(mdtraj_unwrap)

In [ ]:
def msd(trj, atomlist, delta_t):
    """
    Calculate the mean square displacement
    trj: trajectory
    atomlist: list of atoms to calculate the MSD
    delta_t: time step of the trajectory (= log interval * timestep of the integrator)
    """
    time = [ delta_t * i for i in range(len(trj)) ]
    time_ary = np.array(time)
    traj_sel = np.zeros((len(time_ary), len(atomlist), 3))
    traj_sel[:, :, :] = np.array([fr.get_positions()[atomlist] for fr in trj])

    n = traj_sel.shape[0]
    n_atoms = traj_sel.shape[1]
    MSD_x = np.zeros(n)
    MSD_y = np.zeros(n)
    MSD_z = np.zeros(n)
    n_frame = np.zeros(n)
    for at_num in range(n_atoms):
        r = traj_sel[:, at_num, :]
        for i in range(n-1):
            for j in range(i, n):
                MSD_x[j-i] += (r[i, 0] - r[j, 0])**2
                MSD_y[j-i] += (r[i, 1] - r[j, 1])**2
                MSD_z[j-i] += (r[i, 2] - r[j, 2])**2
                n_frame[j-i] += 1
    
    time_ary = time_ary - time_ary[0]
    return time_ary, MSD_x / n_frame, MSD_y / n_frame, MSD_z / n_frame

In [ ]:
t_d,msdx_d,msdy_d,msdz_d = msd(mdtraj,          # trajectory
                               [0,1,2,3,4,5],   # target atom index list
                               50,              # delta_t: time step of the trajectory
                            )

This analysis is done on the only five atoms, but very time consuming. Then, we use Fast fourier transfer for the fast computation. 

Again, MSD is expressed as follows

$$
\begin{align}
|r_{i}(t) - r_i(0)|^2   &= \frac{1}{n-t}\sum_{\tau=0}^{n-t-1}(r_i(t+\tau)-r_i(\tau))^2 \nonumber \\
& = \underbrace{\frac{1}{n-t}\sum_{\tau=0}^{n-t-1}(r(t+\tau)^2 + r(\tau)^2)}_{s_1} -  \underbrace{\frac{2}{n-t}\sum_{\tau=0}^{n-t-1} r_i(t+\tau)r_i(\tau)}_{s_2} \nonumber \\
& = s_1 - s_2
\end{align}
$$
The term of $s_1$ is just twice the average of the squares of the time series. The $s_2$ is autocorrelation function, so Fourier tranform can simplify the computation. (detailed discription is under construction.....)

In [ ]:
import math
def AutocorrFFT(x):
    """
    Compute the autocorrelation function using the FFT
    x is a 1D numpy array
    returns the autocorrelation function of x
    """
    N = len(x)
    F = np.fft.fft(x, n=2*N) #2*N because of zero-padding
    PSD = F * np.conj(F) 
    res = np.fft.ifft(PSD)
    res = np.real(res[:N])
    n = N * np.ones(N) - np.arange(N)
    acf = res / n
    return acf

def msd_fft(trj, atomlist, delta_t, ):
    """
    Calculate the mean square displacement using FFT method
    trj: trajectory
    atomlist: list of atoms to calculate the MSD
    delta_t: time step of the trajectory (= log interval * timestep of the integrator)
    """
    time = [ delta_t * i for i in range(len(trj)) ]
    time_ary = np.array(time)

    traj_sel = np.zeros((len(time_ary), len(atomlist), 3))
    traj_sel[:, :, :] = np.array([fr.get_positions()[atomlist] for fr in trj])

    n = traj_sel.shape[0]
    n_atoms = traj_sel.shape[1]
    MSD_x = np.zeros(n)
    MSD_y = np.zeros(n)
    MSD_z = np.zeros(n)

    for at_num in range(n_atoms):
        r = traj_sel[:, at_num, :]
        Dx = [r[i, 0]**2 for i in range(n)]
        Dy = [r[i, 1]**2 for i in range(n)]
        Dz = [r[i, 2]**2 for i in range(n)]
        Dx = np.append(Dx, 0.0)
        Dy = np.append(Dy, 0.0)
        Dz = np.append(Dz, 0.0)

        tem_x = [AutocorrFFT(r[:, 0])]
        tem_y = [AutocorrFFT(r[:, 1])]
        tem_z = [AutocorrFFT(r[:, 2])]
        S2_x = sum(tem_x)
        S2_y = sum(tem_y)
        S2_z = sum(tem_z)
        Qx = 2 * sum(Dx)
        Qy = 2 * sum(Dy)
        Qz = 2 * sum(Dz)
        S1_x = np.zeros(n)
        S1_y = np.zeros(n)
        S1_z = np.zeros(n)

        for m in range(n):
            if m == 0:
                Qx = Qx - Dx[-1] - Dx[n-m]
                Qy = Qy - Dy[-1] - Dy[n-m]
                Qz = Qz - Dz[-1] - Dz[n-m]
            else:
                Qx = Qx - Dx[m-1] - Dx[n-m]
                Qy = Qy - Dy[m-1] - Dy[n-m]
                Qz = Qz - Dz[m-1] - Dz[n-m]
            S1_x[m] = Qx / (n-m)
            S1_y[m] = Qy / (n-m)
            S1_z[m] = Qz / (n-m)

        msd_x_temp = S1_x - 2*S2_x
        msd_y_temp = S1_y - 2*S2_y
        msd_z_temp = S1_z - 2*S2_z

        MSD_x += msd_x_temp
        MSD_y += msd_y_temp
        MSD_z += msd_z_temp

    time_ary = time_ary - time_ary[0]
    return time_ary, MSD_x/n_atoms, MSD_y/n_atoms, MSD_z/n_atoms

In [ ]:
t_fft,msdx_fft,msdy_fft,msdz_fft = msd_fft(mdtraj_unwrap,  # trajectory
                                          [0,1,2,3,4,5],   # target atom index list
                                          50,              # delta_t: time step of the trajectory
                                          )

In [ ]:
# Check the FFT method and direct method
plt.rcParams.update(params)

fig, ax = plt.subplots()
ax.xaxis.set_minor_locator(ticker.AutoMinorLocator(2))
ax.yaxis.set_minor_locator(ticker.AutoMinorLocator(2))

ax.plot(t_fft, msdx_fft, label="FFT")
ax.plot(t_d, msdx_d, label="Direct", linestyle="-.")
ax.set_xlabel("Time (fs)")
ax.set_ylabel("MSD (${\mathrm {\AA}}^2$)")
ax.legend()

In [ ]:
# Averaged by all Argon atoms
t,msdx,msdy,msdz = msd_fft(mdtraj, np.arange(len(mdtraj[0])), 50)
msd_3d = msdx + msdy + msdz

In [ ]:
params = {
    "font.size": 11.5,
    "font.family": "Arial",
    "axes.labelsize": "large",
    "axes.linewidth": 1.2,
    "xtick.major.size": 7,    # major tick size in points
    "xtick.minor.size": 4,     # minor tick size in points
    "xtick.major.width": 1.0,     # major tick width in points
    "xtick.minor.width": 1.0,     # minor tick width in points
    "xtick.major.pad": 5.0,
    "xtick.minor.pad": 5.0,     # distance to the minor tick label in points
    "xtick.direction" : "in",
    "ytick.right" : True,
    "ytick.major.size": 7,     # major tick size in points
    "ytick.minor.size": 4,       # minor tick size in points
    "ytick.major.width": 1,     # major tick width in points
    "ytick.minor.width": 1.0,    # minor tick width in points
    "ytick.major.pad": 5,     # distance to major tick label in points
    "ytick.minor.pad": 5,    # distance to the minor tick label in points
    "ytick.direction": "in",
    "legend.fontsize": 10,
    "figure.figsize": (3.0, 2.25),
    "axes.prop_cycle": plt.cycler(color=['#e41a1c', '#377eb8', '#000000', '#984ea3',
                                         '#ff7f00', '#4daf4a', '#F6BE00', '#999999',
                                         '#ffff33', '#a65628', '#8B8000','#f781bf'])
}
plt.rcParams.update(params)

fig, ax = plt.subplots()
ax.xaxis.set_minor_locator(ticker.AutoMinorLocator(2))
ax.yaxis.set_minor_locator(ticker.AutoMinorLocator(2))
ax.plot(t, msd_3d)
ax.set_xlabel("Time (fs)")
ax.set_ylabel("MSD (${\mathrm {\AA}}^2$)")

<div class="alert alert-success">

Q. Calculate the diffusion coefficient by linear regression on MSD (~ $2 \times 10^{-5} \mathrm{cm^2 s^{-1}}$)
</div>

In [ ]:
# Get slope
slope, intercept = np.polyfit(t[100:], msd_3d[100:], 1)

plt.rcParams.update(params)
fig, ax = plt.subplots()
ax.xaxis.set_minor_locator(ticker.AutoMinorLocator(2))
ax.yaxis.set_minor_locator(ticker.AutoMinorLocator(2))
ax.plot(t, msd_3d, linewidth=2.5)
ax.plot(t[100:], slope*t[100:] + intercept, linestyle="--", color="gray")
ax.set_xlabel("Time (fs)")
ax.set_ylabel("MSD (${\mathrm {\AA}}^2$)")

<div class="alert alert-block alert-info"> 
<b>NOTE</b>   

This calculation cannot evaluate the standard error of self-diffusion coefficient.  
To evaluate the error, I recommend to use [kinisi](https://github.com/kinisi-dev/kinisi).  

Software: [https://github.com/kinisi-dev/kinisi](https://github.com/kinisi-dev/kinisi)   
Paper: [https://pubs.acs.org/doi/10.1021/acs.jctc.4c01249](https://pubs.acs.org/doi/10.1021/acs.jctc.4c01249)

</div>

### 4.3 Velocity autocorrelation function + Vibrational density of states <a id='vacf'></a>

Self-diffusion coefficient can be calculated by other method (Eq.(2)).  

$$
\begin{align}
D_{\rm{self}} &= \frac{1}{2dt} \lim_{t \to \infty} \langle |\boldsymbol{r}(t) - \boldsymbol{r}(0)|^2 \rangle  \\
& = \frac{1}{d} \int_0^{\infty} \langle \boldsymbol{v}(t) \cdot \boldsymbol{v}(0) \rangle \mathrm{d}t
\end{align}
$$

The integrand, $\langle \boldsymbol{v}(t) \cdot \boldsymbol{v}(0) \rangle$ is called "velocity autocorrelation function".  
This equation is also well knowm as the "Green-Kubo formula". 

In [ ]:
def velacf_fft(mdtraj, atomlist, delta_t):
    """
    Calculate the velocity autocorrelation function using FFT method
    mdtraj: trajectory
    atomlist: list of atoms to calculate the VACF
    delta_t: time step of the trajectory (= log interval * timestep of the integrator)
    """
    time = [ delta_t * i for i in range(len(mdtraj)) ]

    # atomlist = np.arange(len(mdtraj[0].get_positions()))
    time_ary = np.array(time)
    traj_sel = np.zeros((len(time_ary), len(atomlist), 3))
    traj_sel[:, :, :] = np.array([fr.get_velocities()[atomlist] for fr in mdtraj])
    n_atoms = len(atomlist)

    vtv0_x_avg = np.zeros(len(time_ary))
    vtv0_y_avg = np.zeros(len(time_ary))
    vtv0_z_avg = np.zeros(len(time_ary))
    for at_num in range(n_atoms):
        v = traj_sel[:, at_num,:]
        vtv0_x_avg += AutocorrFFT(v[:, 0])
        vtv0_y_avg += AutocorrFFT(v[:, 1])
        vtv0_z_avg += AutocorrFFT(v[:, 2])

    vtv0_x_avg /= n_atoms
    vtv0_y_avg /= n_atoms
    vtv0_z_avg /= n_atoms
    time_ary = time_ary - time_ary[0]
    return time_ary, vtv0_x_avg, vtv0_y_avg, vtv0_z_avg

In [ ]:
time_ary, vtv0_x_avg, vtv0_y_avg, vtv0_z_avg = velacf_fft(mdtraj, np.arange(len(mdtraj[0])), 50)
vtv0_x_avg_Afs = vtv0_x_avg * units.fs * units.fs
vtv0_y_avg_Afs = vtv0_y_avg * units.fs * units.fs
vtv0_z_avg_Afs = vtv0_z_avg * units.fs * units.fs

In [ ]:
plt.plot(time_ary, (vtv0_x_avg_Afs+vtv0_y_avg_Afs+vtv0_z_avg_Afs)/3)
plt.xlabel("Time (fs)")
plt.ylabel("$\langle v(t) \\cdot v(0) \\rangle$ ($\mathrm{\AA}^2$ fs$^{-2}$)")

In [ ]:
plt.xscale("log")
plt.plot(time_ary, (vtv0_x_avg_Afs+vtv0_y_avg_Afs+vtv0_z_avg_Afs)/3)
plt.xlabel("Time (fs)")
plt.ylabel("$\langle v(t) \\cdot v(0) \\rangle$ ($\mathrm{\AA}^2$ fs$^{-2}$)")

In [ ]:
plt.plot(time_ary[:30], (vtv0_x_avg_Afs[:30]+vtv0_y_avg_Afs[:30]+vtv0_z_avg_Afs[:30])/3)
plt.scatter(time_ary[:30], (vtv0_x_avg_Afs[:30]+vtv0_y_avg_Afs[:30]+vtv0_z_avg_Afs[:30])/3)
plt.xlabel("Time (fs)")
plt.ylabel("$\langle v(t) \\cdot v(0) \\rangle$ ($\mathrm{\AA}^2$ fs$^{-2}$)")

<div class="alert alert-success">

Q. Numerically demonstrate that $ D_{\textrm{self}}= \frac{1}{d} \int_0^{\infty} \langle \boldsymbol{v}(t) \cdot \boldsymbol{v}(0) \rangle \mathrm{d}t = \frac{1}{2dt} \lim_{t \to \infty} \langle |\boldsymbol{r}(t) - \boldsymbol{r}(0)|^2 \rangle$
</div>

In [ ]:
from scipy.integrate import trapezoid

# please google the function of "from scipy.integrate import trapezoid"

One can obtain the power spectrum of vibrations (= Phonon DoS (PDoS) = Vibrational DoS (VDoS) = Power spectrum of the atomic velocities) from velocity autocorrelation function. / DoS = Density of States  
This power spectrum help to understand the vibrational property, heat capacity, solvation structure, and so on.  

First, we start from the mass-weighted velocity autocorrelation function:  

$$
C(t) = \sum_{j=1}^N m_j \langle \boldsymbol{v}(t) \cdot \boldsymbol{v}(0) \rangle .
$$

The Fourier transform of $C(t)$ yields the VDoS:  

$$
DoS(\nu) = \frac{2}{k_{B}T} \lim_{\tau \to \infty} \int_{-\tau}^{\tau} C(t) \exp(-i2\pi\nu t) \mathrm{d}t
$$

The density of states at zero frequency, $DoS(0)$, is associated with the self-diffusion constant $D_{\mathrm{self}}$ through:  

$$
D_{\mathrm {self}} = \frac{DoS(0)k_{B}T}{12Mm}  ,
$$

where M is the number of equivalent particles and m is their mass. In addition, the integration of $DoS(\nu)$ over positive frequencies gives the total number of degrees of freedom (3*N*) of the system, 

$$
\int_0^{\infty} DoS(\nu) \mathrm{d}\nu = 3N .
$$

The detailed derivation is given in the following reference 2. 
  
<u>References</u>
- Heat capacity calculation
  1. [Force Field Benchmark of Organic Liquids: Density, Enthalpy of Vaporization, Heat Capacities, Surface Tension, Isothermal Compressibility, Volumetric Expansion Coefficient, and Dielectric Constant](https://pubs.acs.org/doi/10.1021/ct200731v)
  2. [The two-phase model for calculating thermodynamic properties of liquids from molecular dynamics: Validation for the phase diagram of Lennard-Jones fluids](https://doi.org/10.1063/1.1624057)
  3. [Thermodynamics and quantum corrections from molecular dynamics for liquid water](https://pubs.aip.org/aip/jcp/article/79/5/2375/457415/Thermodynamics-and-quantum-corrections-from)
- 

In [ ]:
def calc_vdos(mdtraj, atomlist, delta_t, temperature_K):
    """
    Calculate the velocity density of states
    mdtraj: trajectory
    atomlist: list of atoms to calculate the VDOS
    delta_t: time step of the trajectory (= log interval * timestep of the integrator)
    """
    natoms = len(atomlist)
    vtraj_x = np.zeros([natoms, len(mdtraj)])
    vtraj_y = np.zeros([natoms, len(mdtraj)])
    vtraj_z = np.zeros([natoms, len(mdtraj)])

    for k, atoms in enumerate(mdtraj):
        vtraj_x[:, k] = atoms.get_velocities()[atomlist,0].flatten()  * units.s / units.m  # m/s
        vtraj_y[:, k] = atoms.get_velocities()[atomlist,1].flatten()  * units.s / units.m  # m/s
        vtraj_z[:, k] = atoms.get_velocities()[atomlist,2].flatten()  * units.s / units.m  # m/s

    kT = units.kB/units.J * temperature_K
    beta = 1.0/kT
    N = len(mdtraj)
    fftraj_x = np.fft.fft(vtraj_x, axis=1, n=2*N)
    fdos_x = fftraj_x * np.conjugate(fftraj_x) * 2 * beta * delta_t * 10**-15
    fdos_x = fdos_x[:,:] / N

    fftraj_y = np.fft.fft(vtraj_y, axis=1, n=2*N)
    fdos_y = fftraj_y * np.conjugate(fftraj_y) * 2 * beta * delta_t * 10**-15
    fdos_y = fdos_y[:,:] / N

    fftraj_z = np.fft.fft(vtraj_z, axis=1, n=2*N)
    fdos_z = fftraj_z * np.conjugate(fftraj_z) * 2 * beta * delta_t * 10**-15
    fdos_z = fdos_z[:,:] / N
    freqvals = np.fft.fftfreq(2*N, d=50*1e-15) 

    masses = mdtraj[0].get_masses() * units._amu
    for i, atom_i in enumerate(fdos_x):
        fdos_x[i] = fdos_x[i] * masses[i]
        fdos_y[i] = fdos_y[i] * masses[i]
        fdos_z[i] = fdos_z[i] * masses[i]

    freqvals_invcm = freqvals*1/(units._c * 100)
    return freqvals_invcm[:N], np.abs(fdos_x[:,:N]), np.abs(fdos_y[:,:N]), np.abs(fdos_z[:,:N])

In [ ]:
f, dx, dy, dz = calc_vdos(mdtraj, np.arange(len(mdtraj[0])), 50, 94.4)

In [ ]:
plt.xlabel("Frequency (cm$^{-1}$)")
plt.ylabel("Vibrational DOS (s$^{-1}$)")
plt.plot(f, np.sum(dx, axis=0)+np.sum(dy, axis=0)+np.sum(dz, axis=0))